# Домашнее задание 4. Применение модели акустической диагностики — Python

Восемь заданий: загрузка сохранённого конвейера, извлечение его частей, нормализация,
диагностика одного окна и полного сигнала, эталонные смеси и WAV-файл.

Данные лежат в `data`. Не меняйте имена результирующих переменных.

шкала 0–100 — учебная шкала коэффициентов смешивания, а не физический процент износа.

## Подготовка

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.io import loadmat, wavfile
from scipy.signal import hilbert
from scipy.stats import skew, kurtosis

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

DATA = Path("data")

In [2]:
FEATURE_NAMES = [
    "mean", "median", "rms", "std", "variance", "maximum", "minimum", "range",
    "skewness", "kurtosis", "crest_ratio", "impulse_ratio", "shape_ratio", "time_energy",
    "spectral_max", "dominant_frequency", "spectral_centroid", "spectral_width",
    "spectral_entropy", "spectral_flatness", "mean_frequency", "median_frequency",
    "rolloff_85", "spectral_energy", "spectral_std", "spectral_skewness",
    "spectral_kurtosis",
    *[f"band_{low}_{low + 500}" for low in range(0, 5000, 500)],
    "envelope_spectral_max", "envelope_frequency", "envelope_spectral_energy",
    "envelope_entropy", "envelope_centroid", "envelope_mean", "envelope_std",
    "envelope_max", "envelope_kurtosis", "zero_crossing_rate", "high_mid_low_ratio",
    "envelope_energy_ratio", "absolute_crest_ratio", "fraction_below_q25",
    "small_to_large_amplitude", "small_to_large_energy", "q75_to_q25",
]


def extract_features(signal, fs=10_000):
    """54 time, spectrum and envelope features; one row per 0.4 s window."""
    x = np.asarray(signal, dtype=float)
    centered = x - x.mean()
    absolute = np.abs(centered)
    rms = np.sqrt(np.mean(centered ** 2))
    abs_mean = absolute.mean()
    eps = 1e-10

    time_features = [
        x.mean(), np.median(x), rms, centered.std(ddof=1),
        centered.var(ddof=1), x.max(), x.min(), np.ptp(x),
        skew(centered), kurtosis(centered), x.max() / (rms + eps),
        x.max() / (abs_mean + eps), rms / (abs_mean + eps),
        np.sum(centered ** 2),
    ]

    amplitude = np.abs(np.fft.rfft(centered))[:-1]  # exclude the Nyquist bin as in source
    frequency = np.fft.rfftfreq(len(x), d=1 / fs)[:-1]
    weights = amplitude / (amplitude.sum() + eps)
    cumulative = np.cumsum(weights)
    centroid = np.sum(frequency * weights)
    positive = weights[weights > 0]
    flatness = np.exp(np.mean(np.log(positive))) / (weights.mean() + eps)
    spectral_features = [
        amplitude.max(), frequency[np.argmax(amplitude)], centroid,
        np.sqrt(np.sum((frequency - centroid) ** 2 * weights)),
        -np.sum(positive * np.log(positive)), flatness, centroid,
        frequency[min(np.searchsorted(cumulative, 0.5), len(frequency) - 1)],
        frequency[min(np.searchsorted(cumulative, 0.85), len(frequency) - 1)],
        np.sum(amplitude ** 2), amplitude.std(ddof=1),
        skew(amplitude), kurtosis(amplitude),
    ]

    band_energy = np.array([
        np.sum(amplitude[(frequency >= low) & (frequency < low + 500)] ** 2)
        for low in range(0, 5000, 500)
    ])

    envelope = np.abs(hilbert(centered))
    env_amplitude = np.abs(np.fft.rfft(envelope - envelope.mean()))[:-1]
    env_weights = env_amplitude / (env_amplitude.sum() + eps)
    env_positive = env_weights[env_weights > 0]
    envelope_features = [
        env_amplitude.max(), frequency[np.argmax(env_amplitude)],
        np.sum(env_amplitude ** 2),
        -np.sum(env_positive * np.log(env_positive)),
        np.sum(frequency * env_weights),
    ]

    q25, _, q75 = np.quantile(absolute, [0.25, 0.5, 0.75])
    small = absolute[absolute < q25]
    large = absolute[absolute > q75]
    high_energy = band_energy[7:].sum()
    mid_energy = band_energy[3:7].sum()
    low_energy = band_energy[:3].sum()
    extra = [
        envelope.mean(), envelope.std(ddof=1), envelope.max(), kurtosis(envelope),
        np.count_nonzero(np.diff(np.sign(centered))) / len(x),
        (high_energy + mid_energy) / (low_energy + eps),
        np.sum(env_amplitude ** 2) / (np.sum(amplitude ** 2) + eps),
        absolute.max() / (rms + eps),
        len(small) / len(x),
        small.mean() / (large.mean() + eps),
        np.sum(small ** 2) / (np.sum(large ** 2) + eps),
        q75 / (q25 + eps),
    ]
    result = np.asarray(time_features + spectral_features + band_energy.tolist()
                        + envelope_features + extra, dtype=float)
    assert len(result) == len(FEATURE_NAMES) == 54
    return result

In [3]:
def segment_signal(signal, length=4_000, overlap=0.75):
    """Split a one-dimensional signal into overlapping windows."""
    x = np.asarray(signal, dtype=float).squeeze()
    if x.ndim != 1:
        raise ValueError("signal must be one-dimensional")
    if not 0 <= overlap < 1:
        raise ValueError("overlap must be in [0, 1)")
    step = int(length * (1 - overlap))
    if step < 1 or len(x) < length:
        raise ValueError("invalid window parameters")
    return np.stack([x[start:start + length]
                     for start in range(0, len(x) - length + 1, step)])


def read_wav(path):
    """Read mono/stereo PCM WAV and return a mono float signal."""
    rate, signal = wavfile.read(path)
    original_dtype = signal.dtype
    signal = np.asarray(signal, dtype=float)
    if signal.ndim == 2:
        signal = signal.mean(axis=1)
    if np.issubdtype(original_dtype, np.integer):
        signal = signal / np.iinfo(original_dtype).max
    return rate, signal


def state_label(value):
    """Convert the educational 0–100 model scale into an action category."""
    if value < 10:
        return "идеальное"
    if value < 20:
        return "хорошее"
    if value < 40:
        return "средний износ"
    if value < 70:
        return "сильный износ"
    return "критический износ"

In [4]:
raw = loadmat(DATA / "Bearing.mat")
normal = raw["normal"].ravel().astype(float)
roller = raw["roller"].ravel().astype(float)
inner = raw["inner"].ravel().astype(float)
outer = raw["outer"].ravel().astype(float)
print("Длины:", [len(x) for x in (normal, roller, inner, outer)])

Длины: [120000, 120000, 120000, 120000]


In [5]:
def check(task, value):
    try:
        if task == 1:
            ok = (isinstance(value, dict)
                  and {"models", "scaler", "feature_names", "fs",
                       "segment_length", "overlap", "label_order"} <= set(value))
        elif task == 2:
            models_, scaler_, names_, fs_, length_, overlap_, order_ = value
            ok = (set(models_) == {"inner", "outer", "roller"}
                  and len(scaler_.mean_) == len(names_) == 54
                  and fs_ == 10_000 and length_ == 4_000
                  and np.isclose(overlap_, 0.75)
                  and list(order_) == ["inner", "outer", "roller"])
        elif task == 3:
            arr = np.asarray(value, dtype=float)
            expected = scaler.transform(
                extract_features(normal[:segment_length], fs).reshape(1, -1))
            ok = arr.shape == (1, 54) and np.allclose(arr, expected)
        elif task == 4:
            got = np.array([value[key] for key in label_order], dtype=float)
            ok = set(value) == set(label_order) and np.allclose(
                got, [0.0, 0.01416224, 0.27519833], atol=0.02)
        elif task == 5:
            got = np.array([value[key] for key in label_order], dtype=float)
            ok = set(value) == set(label_order) and np.allclose(
                got, [3.5, 99.5, 5.1], atol=0.15)
        elif task == 6:
            got = np.array([value[key] for key in label_order], dtype=float)
            ok = np.allclose(got, [0.2, 0.1, 0.4], atol=0.11)
        elif task == 7:
            got = np.array([value[key] for key in label_order], dtype=float)
            ok = np.allclose(got, [4.6, 3.0, 9.2], atol=0.11)
        elif task == 8:
            got = np.array([value[key] for key in label_order], dtype=float)
            ok = np.allclose(got, [4.2, 2.8, 0.4], atol=0.11)
        else:
            raise ValueError("неизвестный номер")
    except Exception as exc:
        print(f"❌ Задание {task}: {type(exc).__name__}: {exc}")
        return
    print(f"{'✅' if ok else '❌'} Задание {task}" +
          (" выполнено" if ok else ": проверьте результат"))

## Задание 1. Загрузить комплект модели

Загрузите `DATA / "bearing_models_python.joblib"` через `joblib.load` и сохраните словарь в `bundle`.
Загружать joblib/pickle можно только из доверенного источника.

In [6]:
model_path = DATA.joinpath("bearing_models_python.joblib")
bundle = joblib.load(model_path)
print(sorted(bundle))

['feature_names', 'fs', 'label_order', 'models', 'overlap', 'scaler', 'segment_length']


In [7]:
check(1, bundle)

✅ Задание 1 выполнено


## Задание 2. Извлечь составные части

Извлеките ключи `models`, `scaler`, `feature_names`, `fs`, `segment_length`, `overlap`,
`label_order` в одноимённые переменные.

In [8]:
models, scaler, feature_names = (bundle[name] for name in ("models", "scaler", "feature_names"))
fs, segment_length, overlap, label_order = (
    bundle[name] for name in ("fs", "segment_length", "overlap", "label_order")
)
print("Количество признаков:", len(feature_names))

Количество признаков: 54


In [9]:
check(2, (models, scaler, feature_names, fs, segment_length, overlap, label_order))

✅ Задание 2 выполнено


## Задание 3. Нормализовать признаки

Напишите `normalize_features(features)`. Преобразуйте вход в float-массив формы `(1, -1)`
и примените **уже обученный** `scaler.transform`. Не используйте `fit` на новых данных.

In [10]:
def normalize_features(features):
    row = np.array(features, dtype=float).reshape(1, -1)
    normalized = scaler.transform(row)
    return normalized

first_window = normal[:segment_length]
normalized_example = normalize_features(extract_features(first_window, fs))
print(normalized_example.shape)

(1, 54)


In [11]:
check(3, normalized_example)

✅ Задание 3 выполнено


## Задание 4. Диагностика одного сегмента

Напишите `diagnose_segment(signal, sample_rate=fs)`: вычислите 54 признака,
нормализуйте их и верните словарь `{key: prediction}` для ключей из `label_order`.

In [12]:
def diagnose_segment(signal, sample_rate=fs):
    row = normalize_features(extract_features(signal, fs=sample_rate))
    estimates = {}
    for component in label_order:
        estimates[component] = float(models[component].predict(row).item())
    return estimates

segment_result = diagnose_segment(normal[:segment_length])
print(segment_result)

{'inner': 0.0, 'outer': 0.014162236400543418, 'roller': 0.27519832517148163}


In [13]:
check(4, segment_result)

✅ Задание 4 выполнено


## Задание 5. Диагностика полного сигнала

Напишите `diagnose_signal(signal, sample_rate=fs)`:

1. разделите сигнал через `segment_signal(signal, segment_length, overlap)`;
2. соберите 54 признака каждого окна в матрицу через `np.vstack`;
3. одним вызовом `scaler.transform` нормализуйте всю матрицу;
4. получите вектор прогнозов каждой модели, усредните, ограничьте диапазоном 0–100 и округлите до 0,1.

Самопроверка использует эталонный сигнал внешнего кольца.

In [14]:
def diagnose_signal(signal, sample_rate=fs):
    parts = segment_signal(signal, length=segment_length, overlap=overlap)
    rows = []
    for part in parts:
        rows.append(extract_features(part, fs=sample_rate))
    matrix = scaler.transform(np.vstack(rows))
    estimates = {}
    for component in label_order:
        predictions = models[component].predict(matrix)
        average = np.mean(predictions)
        estimates[component] = round(float(np.clip(average, 0.0, 100.0)), 1)
    return estimates

outer_result = diagnose_signal(outer)
print(outer_result)

{'inner': 3.5, 'outer': 99.5, 'roller': 5.1}


In [15]:
check(5, outer_result)

✅ Задание 5 выполнено


## Задание 6. Исправный подшипник

Вызовите `diagnose_signal(normal, fs)` и сохраните словарь в `ans6`.

In [16]:
ans6 = diagnose_signal(signal=normal, sample_rate=fs)
print(ans6)

{'inner': 0.2, 'outer': 0.1, 'roller': 0.4}


In [17]:
check(6, ans6)

✅ Задание 6 выполнено


## Задание 7. Комбинированный сигнал

Создайте смесь `normal` с коэффициентами 0,08 для `inner`, 0,07 для `outer`
и 0,12 для `roller` по формуле `normal + a * (defect - normal)`. Диагностируйте смесь и сохраните словарь в `ans7`.

In [18]:
mixture = normal.copy()
for coefficient, reference in zip((0.08, 0.07, 0.12), (inner, outer, roller)):
    mixture += coefficient * (reference - normal)
ans7 = diagnose_signal(signal=mixture, sample_rate=fs)
print(ans7)

{'inner': 4.6, 'outer': 3.0, 'roller': 9.2}


In [19]:
check(7, ans7)

✅ Задание 7 выполнено


## Задание 8. WAV-файл

Прочитайте `DATA / "Подшипник.wav"` через готовую функцию `read_wav`, передайте сигнал
и его частоту в `diagnose_signal` и сохраните словарь в `ans8`.

In [20]:
wav_fs, wav_signal = read_wav(DATA.joinpath("Подшипник.wav"))
ans8 = diagnose_signal(signal=wav_signal, sample_rate=wav_fs)
print(ans8)

{'inner': 4.2, 'outer': 2.8, 'roller': 0.4}


In [21]:
check(8, ans8)

✅ Задание 8 выполнено
